In [1]:
!wget https://raw.githubusercontent.com/karpathy/ng-video-lecture/master/input.txt


--2026-04-29 06:18:20--  https://raw.githubusercontent.com/karpathy/ng-video-lecture/master/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.006s  

2026-04-29 06:18:20 (174 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [2]:
with open('input.txt' , 'r',  encoding='utf-8') as f:
  text=f.read()

In [3]:
print("Length of dataset:",len(text))

Length of dataset: 1115394


In [4]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [5]:
chars = sorted(list(set(text)))
vocab_size=len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [6]:
#create a mapping from characters to integers
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s : [stoi[c] for c in s] #Taking a string and output list of integers
decode = lambda l : ''.join([itos[i] for i in l]) #Taking a list of integers output a string

print(encode("Hii There"))
print(decode(encode("Hii There")))

[20, 47, 47, 1, 32, 46, 43, 56, 43]
Hii There


In [7]:
import torch
data = torch.tensor(encode(text),dtype=torch.long)
print(data.shape,data.dtype)
print(data[:1000])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [8]:
n= int(0.9*len(data))
train_data=data[:n]
val_data=data[n:]

In [9]:
block_size= 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [10]:
x=train_data[:block_size]
y=train_data[1:block_size+1]
for t in range(block_size):
  context=x[:t+1]
  target=y[t]
  print(f"when input is {context} the target:{target}")

when input is tensor([18]) the target:47
when input is tensor([18, 47]) the target:56
when input is tensor([18, 47, 56]) the target:57
when input is tensor([18, 47, 56, 57]) the target:58
when input is tensor([18, 47, 56, 57, 58]) the target:1
when input is tensor([18, 47, 56, 57, 58,  1]) the target:15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target:47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target:58


In [11]:
torch.manual_seed(1337)
batch_size=4
block_size=8

def get_batch(split):
  #Generating a small batch data of inputs x and targets y
  data=train_data if split=='train' else val_data
  ix=torch.randint(len(data)-block_size,(batch_size,))
  x=torch.stack([data[i:i+block_size]for i in ix])
  y=torch.stack([data[i+1:i+block_size+1]for i in ix])
  return x,y

xb,yb=get_batch('train')
print('inputs')
print(xb)
print('targets')
print(yb.shape)
print(yb)
print('------')

for b in range(batch_size):
  for t in range(block_size):
    context=xb[b,:t+1]
    target=yb[b,t]
    print(f"When input is {context.tolist()} target is {target}")

inputs
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
------
When input is [24] target is 43
When input is [24, 43] target is 58
When input is [24, 43, 58] target is 5
When input is [24, 43, 58, 5] target is 57
When input is [24, 43, 58, 5, 57] target is 1
When input is [24, 43, 58, 5, 57, 1] target is 46
When input is [24, 43, 58, 5, 57, 1, 46] target is 43
When input is [24, 43, 58, 5, 57, 1, 46, 43] target is 39
When input is [44] target is 53
When input is [44, 53] target is 56
When input is [44, 53, 56] target is 1
When input is [44, 53, 56, 1] target is 58
When input is [44, 53, 56, 1, 58] target is 46
When input is [44, 53, 56, 1, 58, 46] target is 39
When input is [

In [12]:
print(xb) # input for transformer

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


In [13]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

  def __init__(self,vocab_size):
    super().__init__()
    #Each token directly reads the logits of the next token from a lookup table
    self.token_embedding_table = nn.Embedding(vocab_size,vocab_size)

  def forward(self,idx,targets=None):

    #idx and targets are bith (B,T) tensor of integers
    logits=self.token_embedding_table(idx) # (B,T,C) Batch Time Channel
    #Reshaping the logits beacause Cross Entropy needs B C T not B T C
    if targets is None:
      loss=None
    else:
      B,T,C=logits.shape
      logits=logits.view(B*T,C) #Creating the array 2  Dimensional
      targets=targets.view(B*T)

      loss = F.cross_entropy(logits,targets)

    return logits , loss

  def generate(self,idx,max_new_tokens):
    #idx is (B,T) arrays of indices in current context
    for _ in range(max_new_tokens):
      #get the prediction
      logits,loss=self(idx)
      #focusing only on the last time step
      logits=logits[:,-1,:] #become (B,C)
      #Applying softmax to get probabilities
      probs=F.softmax(logits,dim=1) # (B,C)
      #sample from the distribution
      idx_next= torch.multinomial(probs,num_samples=1) #(B,1)
      #append sampled index to the running sequence
      idx=torch.cat((idx,idx_next),dim=1) # (B,T+1)
    return idx

m=BigramLanguageModel(vocab_size)
logits,loss=m(xb,yb)
print(logits.shape)
print(loss)
print(decode(m.generate(torch.zeros((1,1),dtype=torch.long) ,max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [14]:
#create a pytorch optimizer
optimizer = torch.optim.AdamW(m.parameters(),lr=1e-3)


In [15]:
batch_size=32
for steps in range(10000):

  #Sample of batch of data
  xb,yb=get_batch('train')

  #evaluate the loss
  logits,loss=m(xb,yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()

print(loss.item())

2.382369041442871


In [16]:
print(decode(m.generate(torch.zeros((1,1),dtype=torch.long) ,max_new_tokens=500)[0].tolist()))


lso br. ave aviasurf my, yxMPZI ivee iuedrd whar ksth y h bora s be hese, woweee; the! KI 'de, ulseecherd d o blllando;LUCEO, oraingofof win!
RIfans picspeserer hee tha,
TOFonk? me ain ckntoty ded. bo'llll st ta d:
ELIS me hurf lal y, ma dus pe athouo
BEY:! Indy; by s afreanoo adicererupa anse tecorro llaus a!
OLeneerithesinthengove fal amas trr
TI ar I t, mes, n IUSt my w, fredeeyove
THek' merer, dd
We ntem lud engitheso; cer ize helorowaginte the?
Thak orblyoruldvicee chot, p,
Bealivolde Th li


The mathematical trick of self Attention

In [17]:
#Lets consider a toy example
torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [18]:
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # (t,C)
        xbow[b,t] = torch.mean(xprev, 0)

In [19]:
#Version 2 using matrix
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x # (B, T, T) @ (B, T, C) ----> (B, T, C)
torch.allclose(xbow, xbow2, atol=1e-6)

True

In [20]:
#Version 3 using softmax

tril=torch.tril(torch.ones(T, T))
wei=torch.zeros((T,T))
wei=wei.masked_fill(tril==0,float('-inf'))
wei=F.softmax(wei,dim=1)
xbow3=wei @ x
torch.allclose(xbow, xbow3,atol=1e-6)

True

In [21]:
#Getting average but using matrices and not for loop
torch.manual_seed(42)
a=torch.tril(torch.ones(3,3))
a=a/torch.sum(a,1,keepdim=True)
b=torch.randint(0,10,(3,2)).float()
c=a@b
print("a=")
print(a)
print('--')
print("b=")
print(b)
print('--')
print("c=")
print(c)
print('--')

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])
--


In [28]:
#Version 4 Self Attention
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

#Single Head of self Attention
head_size=16
key =nn.Linear(C,head_size,bias=False)
query= nn.Linear(C,head_size,bias=False)
value= nn.Linear(C,head_size,bias=False)
k=key(x) # B,T,16
q=query(x) # B,T,16

wei= q @ k.transpose(-2,-1) # (B,T,16) @ (B,16,T) ---> (B,T,T)

tril=torch.tril(torch.ones(T, T))
#wei=torch.zeros((T,T))
wei=wei.masked_fill(tril==0,float('-inf'))
wei=F.softmax(wei,dim=-1)
v=value(x)
out =wei @ v
# out =wei @ x

out.shape

torch.Size([4, 8, 16])

In [29]:
k= torch.randn(B,T,head_size)
q= torch.randn(B,T,head_size)
wei= q @ k.transpose(-2,-1)* head_size**-0.5

In [30]:
class LayerNorm1d: # (used to be BatchNorm1d)

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

  def __call__(self, x):
    # calculate the forward pass
    xmean = x.mean(1, keepdim=True) # batch mean
    xvar = x.var(1, keepdim=True) # batch variance
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    return self.out

  def parameters(self):
    return [self.gamma, self.beta]

torch.manual_seed(1337)
module = LayerNorm1d(100)
x = torch.randn(32, 100) # batch size 32 of 100-dimensional vectors
x = module(x)
x.shape

torch.Size([32, 100])

In [32]:
x[:,0].mean(),x[:,0].std()

(tensor(0.1469), tensor(0.8803))

In [34]:
x[0,:].mean(),x[0,:].std()

(tensor(-9.5367e-09), tensor(1.0000))

Full code


In [35]:
from altair import value

import torch
import torch.nn as nn
from torch.nn import functional as F

#hyperparameters
batch_size = 64 #how many independent sequences will we process in parallel?
block_size = 256 #what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters=200
n_embd=384
n_head=6
n_layer=6
dropout=0.2



torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/ng-video-lecture/master/input.txt
with open('input.txt' , 'r',  encoding='utf-8') as f:
  text=f.read()


chars = sorted(list(set(text)))
vocab_size=len(chars)
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s : [stoi[c] for c in s] #Taking a string and output list of integers
decode = lambda l : ''.join([itos[i] for i in l]) #Taking a list of integers output a string

#Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

#data loading

def get_batch(split):
  #Generating a small batch data of inputs x and targets y
  data=train_data if split=='train' else val_data
  ix=torch.randint(len(data)-block_size,(batch_size,))
  x=torch.stack([data[i:i+block_size]for i in ix])
  y=torch.stack([data[i+1:i+block_size+1]for i in ix])
  x,y=x.to(device),y.to(device)
  return x,y

@torch.no_grad()
def estimate_loss():
    out={}
    model.eval()
    for split in ['train','val']:
        losses=torch.zeros(eval_iters)
        for k in range(eval_iters):
            X,Y=get_batch(split)
            logits,loss=model(X,Y)
            losses[k]=loss.item()
        out[split]=losses.mean()
    model.train()
    return out

class Head(nn.Module):
  """One head of Self Attention"""
  def __init__(self,head_size):
    super().__init__()
    self.key = nn.Linear(n_embd,head_size,bias=False)
    self.query = nn.Linear(n_embd,head_size,bias=False)
    self.value = nn.Linear(n_embd,head_size,bias=False)
    self.register_buffer('tril',torch.tril(torch.ones(block_size,block_size)))

    self.dropout=nn.Dropout(dropout)

  def forward(self,x):
    B,T,C = x.shape
    k= self.key(x) # B,T,16
    q= self.query(x) # B,T,16

    wei= q @ k.transpose(-2,-1) * C**-0.5 # (B,T,16) @ (B,16,T) ---> (B,T,T)
    wei=wei.masked_fill(self.tril[:T,:T]==0,float('-inf'))
    wei=F.softmax(wei,dim=-1)
    wei=self.dropout(wei)
    v=self.value(x)
    out =wei @ v
    return out

class MultiHeadAttention(nn.Module):
  """Multiple heads of Self Attention in parallel"""

  def __init__(self,n_head,head_size):
    super().__init__()
    self.heads= nn.ModuleList([Head(head_size) for _ in range(n_head)])
    self.proj=nn.Linear(n_embd,n_embd)
    self.dropout=nn.Dropout(dropout)

  def forward(self,x):
    out= torch.cat([h(x) for h in self.heads],dim=-1)
    out=self.proj(out)
    return out

class FeedForward(nn.Module):
  """A simple linear layer followed by a non-linearity"""

  def __init__(self,n_embd):
      super().__init__()
      self.net=nn.Sequential(
         nn.Linear(n_embd,4 * n_embd),
         nn.ReLU(),
         nn.Linear(4 * n_embd,n_embd),
         nn.Dropout(dropout)
      )

  def forward(self,x):
      return self.net(x)



class LayerNorm: # (used to be BatchNorm1d)

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

  def __call__(self, x):
    # calculate the forward pass
    xmean = x.mean(1, keepdim=True) # batch mean
    xvar = x.var(1, keepdim=True) # batch variance
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    return self.out

  def parameters(self):
    return [self.gamma, self.beta]


class Block(nn.Module):
  """Transformer block: communication followed by a feed forward"""

  def __init__(self,n_embd,n_head):
    super().__init__()
    head_size = n_embd//n_head
    self.sa=MultiHeadAttention(n_head,head_size)
    self.ffwd=FeedForward(n_embd)
    self.ln1=nn.LayerNorm(n_embd)
    self.ln2=nn.LayerNorm(n_embd)

  def forward(self,x):
    x= x + self.sa(self.ln1(x))
    x= x + self.ffwd(self.ln2(x))
    return x



#Super simple bigram model
class BigramLanguageModel(nn.Module):

  def __init__(self):
    super().__init__()
    #Each token directly reads the logits of the next token from a lookup table
    self.token_embedding_table = nn.Embedding(vocab_size,n_embd)
    self.position_embedding_table = nn.Embedding(block_size,n_embd)
    self.blocks=nn.Sequential(
    *[Block(n_embd,n_head=n_head) for _ in range(n_layer)]
    )
    self.ln_f=nn.LayerNorm(n_embd) #Final layer norm
    self.lm_head=nn.Linear(n_embd,vocab_size)


  def forward(self,idx,targets=None):
    B,T =idx.shape
    #idx and targets are bith (B,T) tensor of integers
    token_embd=self.token_embedding_table(idx) # (B,T,C) Batch Time Channel
    pos_embd=self.position_embedding_table(torch.arange(T,device=device))
    x=token_embd + pos_embd
    x=self.blocks(x)
    logits=self.lm_head(x)
    #Reshaping the logits beacause Cross Entropy needs B C T not B T C

    if targets is None:
      loss=None
    else:
      B,T,C=logits.shape
      logits=logits.view(B*T,C) #Creating the array 2  Dimensional
      targets=targets.view(B*T)

      loss = F.cross_entropy(logits,targets)

    return logits , loss

  def generate(self,idx,max_new_tokens):
    #idx is (B,T) arrays of indices in current context
    for _ in range(max_new_tokens):
      #croping idx to last block_size tokens
      idx_cond=idx[:,-block_size:]
      #get the prediction
      logits,loss=self(idx_cond)
      #focusing only on the last time step
      logits=logits[:,-1,:] #become (B,C)
      #Applying softmax to get probabilities
      probs=F.softmax(logits,dim=1) # (B,C)
      #sample from the distribution
      idx_next= torch.multinomial(probs,num_samples=1) #(B,1)
      #append sampled index to the running sequence
      idx=torch.cat((idx,idx_next),dim=1) # (B,T+1)
    return idx



model=BigramLanguageModel()
m=model.to(device)

#create a pytorch optimizer
optimizer = torch.optim.AdamW(m.parameters(),lr=1e-3)

for iter in range(max_iters):

    #every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    #sample a batch of data
    xb,yb=get_batch('train')
    #evaluate the loss
    logits,loss=model(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

#generate from the model
context = torch.zeros((1,1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=100)[0].tolist()))

step 0: train loss 4.4753, val loss 4.4709
step 500: train loss 1.7248, val loss 1.8744
step 1000: train loss 1.4025, val loss 1.6232
step 1500: train loss 1.2717, val loss 1.5446
step 2000: train loss 1.1776, val loss 1.5202
step 2500: train loss 1.0926, val loss 1.5446
step 3000: train loss 1.0052, val loss 1.5714
step 3500: train loss 0.9062, val loss 1.6325
step 4000: train loss 0.8074, val loss 1.6996
step 4500: train loss 0.7005, val loss 1.7639

Of you'ld offend and calves it out for so
The white of the war, the herds ;
It is unpily to death: t
